In [ ]:
# @title
import os
os.environ["GROQ_API_KEY"] = "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"

## Install libraries

In [ ]:
!pip install -q youtube-transcript-api langchain-community langchain_groq \
               faiss-cpu tiktoken python-dotenv langchain_huggingface \
               langchain_text_splitters

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

## Step 1a - Indexing (Document Ingestion)

In [ ]:
video_id = "AultJcNb90c" # only the ID, not full URL
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled # Import get_transcript directly
    # If you don’t care which language, this returns the “best” one
transcript_list = YouTubeTranscriptApi().fetch(video_id, languages=['en']) # Call get_transcript directly


In [ ]:
video_id = "bAWujyAl1Kk" # only the ID, not full URL
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled # Import get_transcript directly
try:
    # Request the 'hi' (Hindi) language transcript
    transcript_list = YouTubeTranscriptApi().fetch(video_id, languages=['hi']) # Call get_transcript directly

    # Flatten it to plain text
    transcript = " ".join(chunk.text for chunk in transcript_list)
    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video.")

हाय गाइज़, माय नेम इज नितेश एंड यू आर वेलकम टू माय YouTube चैनल। इस वीडियो में भी हम लोग अपना एजेंटिक एआई यूजिंग लंग ग्राफ प्लेलिस्ट कंटिन्यू करेंगे और वीडियो को स्टार्ट करने के पहले एक बार क्विकली रिवाइज कर लेते हैं कि अभी तक हमने इस प्लेलिस्ट में क्या-क्या कवर किया है। सो अभी तक इस प्लेलिस्ट में मैंने चार वीडियोस डाले हैं और मोस्टली जो भी हमारा डिस्कशन रहा है वो थ्योरेटिकल रहा है, कॉनसेप्चुअल रहा है। सो जो सबसे पहला वीडियो था वहां पे हमने डिस्कस किया था कि व्हाट आर द मेन डिफरेंसेस बिटवीन जनरेटिव एआई एंड एजेंटिक एआई। एक बार जब हमने ये समझ लिया उसके बाद हमने बहुत डिटेल में एक डीप ड्राइव लिया था टू अंडरस्टैंड व्हाट एजेंटिक एआई इज़। ठीक है? हमने एक यूज़ केस लिया था। उस यूज़ केस के अराउंड हमने सारा डिस्कशन किया था। उसके बाद हमने लैंग ग्राफ की तरफ मूव किया और हमने सबसे पहले ये समझने की कोशिश की कि अगर लैंग चेन जैसा फ्रेमवर्क ऑलरेडी एग्ज़िस्ट करता है तो लैंग ग्राफ की जरूरत क्यों पड़ी? और फिर हमने फिर से एक यूज़ केस लेकर के दोनों लाइब्रेरीज के बीच में जो कोर डिफरेंसेस हैं वो समझने की कोशिश की। और फ

## Step 1b - Indexing (Text Splitting)

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [ ]:
len(chunks)

43

In [ ]:
chunks[5]

Document(metadata={}, page_content='यह सारी चीजें आपको लैंग चेन में ही मिलती हैं। तो, हमें आज के वीडियो में लैंग चेन की जरूरत पड़ेगी। और लास्ट में हमें एक और लैंग चेन की डिपेंडेंसी चाहिए जिसको हम लंग चेन ओपन एआई बुलाते हैं। बिकॉज़ हम लोग ओपन एआई के मॉडल्स को यूज़ करेंगे आज के लेक्चर में। और एक और लास्ट लाइब्रेरी हमें चाहिए पिप इंस्टॉल. बिकॉज़ हम एनवायरमेंट वेरिएबल्स के साथ काम करेंगे। तो उनको रीड करने के लिए यह लाइब्रेरी आपका हेल्प करेगी। एंड नाउ वी आर डन। ठीक है? फिलहाल हमें इतनी ही लाइब्रेरीज़ चाहिए। एक काम करते हैं जल्दी से एक बार टेस्ट कर लेते हैं कि जो भी लाइब्रेरीज़ हमने इंस्टॉल की है वो सही से हम लोग इंपोर्ट कर पा रहे हैं कि नहीं। तो, हम एक नई फाइल बना रहे हैं। लेट्स कॉल इट 0 अंडरsर टेस्ट इंस्टॉलेशन। आई पीy एनb। सो बेसिकली हम एक जुपिटर नोटबुक क्रिएट कर रहे हैं। ठीक है? बिकॉज़ आज का सारा जो कोड हम करेंगे वो हम लोग जुपिटर नोटबुक में करेंगे। और वो इसलिए करेंगे बिकॉज़ जुपिटर नोटबुक में आप बहुत आसानी से लंग ग्राफ में बनाए हुए ग्राफ्स को प्रिंट करके देख सकते हो। ठीक है? इसीलिए हम लोग जुपिटर 

## Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")
vector_store = FAISS.from_documents(chunks, embeddings)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
vector_store.index_to_docstore_id

{0: 'ebf2abc7-79d0-491e-af35-60852df12aee',
 1: 'e5eb1a14-f970-4946-b058-da9aaac84767',
 2: '7aeb7635-dd46-4e8e-b809-c2af911900a3',
 3: '085798d9-8c56-4fcf-940b-7823b8b1678f',
 4: '407b95bf-f087-415d-91ad-85b03b5e740b',
 5: 'd4efb95c-8cde-49f5-bdfc-af86e5b75d18',
 6: '0b015229-73f1-4a8e-8fb9-a6424a547495',
 7: '669cd88c-56d9-4044-8ad3-0fe27a1da754',
 8: 'def43fd3-b27a-496f-b2e8-554b07c06c2f',
 9: 'a0021f96-e2bd-4dde-858e-e253f355ee3b',
 10: 'a9a348ce-6d2a-4428-ba5a-4df33f39e1f8',
 11: 'afb588f2-fc66-4a4a-8422-452b35711fbe',
 12: '705d257d-26d6-43dd-8580-1f04e92672a2',
 13: '20b497c2-bef2-4656-8a5a-60fb110f425b',
 14: '4a991d56-b758-448d-b4dd-324415db22d3',
 15: '4061c587-936b-4a7e-919e-21c77830bee2',
 16: 'f78bfaba-60e1-4d94-94df-5a05b27b544e',
 17: '9f9f8446-1d6d-4ad7-94a1-a860656282ba',
 18: '8578af7a-82fd-48b7-a476-8fcfb6c00855',
 19: 'dc6cfd45-7481-4b9c-8a85-0427da10d1f0',
 20: '8fd0a20b-91b3-4631-b35f-f2dde5d7896b',
 21: '20faeb65-a7db-4ab2-b205-56eee8c4b865',
 22: '7090f8de-e914-

In [ ]:
vector_store.get_by_ids(['7c9932ab-f5fe-4902-b6fb-1f8d8413e77e'])

[Document(id='7c9932ab-f5fe-4902-b6fb-1f8d8413e77e', metadata={}, page_content='टाइटल क्या है और करंट आउटलाइन क्या है? एक बार जब ये दोनों चीजें हमारे पास आ जाएंगी तो हम लोग एक प्र्प्ट फिर से बनाएंगे। प्र्प हो जाएगा राइट अ डिटेल्ड ब्लॉग ऑन द टाइटल यूजिंग द फॉलोइंग आउटलाइन स्लश इन यहां पे हम अपना आउटलाइन प्रोवाइड कर देंगे। ठीक है? और फिर हम लोग क्या करेंगे? मॉडल डॉट इनवोक करेंगे। और यहां पर हम अपना प्रप पास कर देंगे। जो भी कंटेंट है वह हम लोग एक्सट्रैक्ट कर लेंगे और उसको कंटेंट बोल के एक वेरिएबल में स्टोर कर लेंगे और स्टेट के कंटेंट में हम कंटेंट को डाल देंगे और हम लोग रिटर्न कर देंगे स्टेट को। ठीक है? तो ये हमारे दोनों नोट्स प्रिपेयर हो गए। नाउ वी हैव टू ऐड द एजेस। सो वी विल राइट ग्राफ डॉट ऐड एज स्टार्ट से स्टार्ट करेंगे और जाएंगे क्रिएट आउटलाइन के पास ग्राफ डॉट ऐड एज क्रिएट आउटलाइन से जाएंगे क्रिएट ब्लॉक के पास ग्राफ डॉट ऐड एच क्रिएट ब्लॉग से जाएंगे एंड के पास। ठीक है? और ग्राफ डॉट कंपाइल करके एक बार चेक कर लेते हैं। ठीक है? स्टार्ट क्रिएट आउटलाइन क्रिएट ब्लॉग एंड। ठीक है? वापस से इसको

## Step 2 - Retrieval

In [ ]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [ ]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7fc7a71ca5a0>, search_kwargs={'k': 4})

In [ ]:
retriever.invoke('what is rag?')

[Document(id='0b015229-73f1-4a8e-8fb9-a6424a547495', metadata={}, page_content='नोटबुक में करेंगे। और वो इसलिए करेंगे बिकॉज़ जुपिटर नोटबुक में आप बहुत आसानी से लंग ग्राफ में बनाए हुए ग्राफ्स को प्रिंट करके देख सकते हो। ठीक है? इसीलिए हम लोग जुपिटर नोटबुक यूज़ कर रहे हैं। आगे जब हम प्रोजेक्ट्स वगैरह बिल्ड करेंगे तो हम लोग नॉर्मल Python फाइल्स में कोड लिखेंगे। तो यहां पे हम लोग सबसे पहले एक बार लिख लेते हैं। फ्रॉम लंग ग्राफ डॉट ग्राफ इंपोर्ट स्टेट ग्राफ एक बार शिफ्ट एंटर प्रेस कर रहा हूं मैं। यहां पे मुझे बोला जा रहा है कि अपना Python एनवायरमेंट सेलेक्ट करो। सो आई विल सेलेक्ट दिस वन जो मैंने अपना वर्चुअल एनवायरमेंट अभी बनाया है। यहीं पर मैंने सारी लाइब्रेरीज इंस्टॉल की हैं। तो एक बार फिर से रन करते हैं। इट सेस कि मुझे यहां पर आई पाई कर्नल पैकेज इंस्टॉल करना पड़ेगा। तो मैं इंस्टॉल कर लेता हूं एंड इंस्टॉल हो गया। और यह पर्टिकुलर सेल चल भी गया। एक बार फिर से देख लेते हैं। फ्रॉम लैंग ग्राफ यू कैन सी सजेशन आ रहा है। लैंगचेन का भी सजेशन आ रहा है। व्हिच मींस हमारा इंस्टॉलेशन सही से काम कर रहा है।

## Step 3 - Augmentation

In [ ]:
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.2)

In [ ]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [ ]:
question          = "is the topic of new birth policy in china discussed? if yes then what was discussed"
retrieved_docs    = retriever.invoke(question)

In [ ]:
retrieved_docs

[Document(id='648c0d1a-6f7e-4a91-85c1-b01c98839dc5', metadata={}, page_content='रेट माय ब्लॉग। ठीक है? और जनरेट अ स्कोर इंटीजर स्कोर। ठीक है? तो आपको स्टेट में चेंजज़ करने पड़ेंगे और आपको वर्क फ्लो में भी थोड़े चेंजज़ करने पड़ेंगे। ये करके देखना आप। अह बहुत खास कुछ इंप्रूवमेंट नहीं है। बट सिंस आप लोग पहली बार कर रहे हो तो ये छोटी-छोटी चीजें भी अगर आप खुद से कर पा रहे हो तो इट इज़ सिग्निफिकेंट प्रोग्रेस। ठीक है? तो अह विथ दैट आई विल कंक्लूड दिस वीडियो। अगर आपको वीडियो पसंद आया तो प्लीज लाइक करना। अगर आपने चैनल को सब्सक्राइब नहीं किया है, प्लीज डू सब्सक्राइब। मिलते हैं नेक्स्ट वीडियो में। बाय।'),
 Document(id='66499d5a-a1eb-4ee0-a846-b8117cc34dcc', metadata={}, page_content='फ्यूचर आउटलुक, कंक्लूजन। अब हम लोग लिखते हैं फाइनल स्टेट कंटेंट प्रिंट। इससे हमें हमारा पूरा ब्लॉक दिखाई देगा। ठीक है? यह ऐसा एक-एक लाइन का इसलिए दिख रहा है बिकॉज़ यह हॉरिजॉन्टली स्क्रबल है। ठीक है? दिस इज़ अ प्रॉपर ब्लॉक जो जनरेट हुआ है। ठीक है? तो आई होप आपको समझ में आ गया कि कैसे आप प्र्ट चेनिंग कर सकते हो यूजिंग लैंग ग्र

In [ ]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

'रेट माय ब्लॉग। ठीक है? और जनरेट अ स्कोर इंटीजर स्कोर। ठीक है? तो आपको स्टेट में चेंजज़ करने पड़ेंगे और आपको वर्क फ्लो में भी थोड़े चेंजज़ करने पड़ेंगे। ये करके देखना आप। अह बहुत खास कुछ इंप्रूवमेंट नहीं है। बट सिंस आप लोग पहली बार कर रहे हो तो ये छोटी-छोटी चीजें भी अगर आप खुद से कर पा रहे हो तो इट इज़ सिग्निफिकेंट प्रोग्रेस। ठीक है? तो अह विथ दैट आई विल कंक्लूड दिस वीडियो। अगर आपको वीडियो पसंद आया तो प्लीज लाइक करना। अगर आपने चैनल को सब्सक्राइब नहीं किया है, प्लीज डू सब्सक्राइब। मिलते हैं नेक्स्ट वीडियो में। बाय।\n\nफ्यूचर आउटलुक, कंक्लूजन। अब हम लोग लिखते हैं फाइनल स्टेट कंटेंट प्रिंट। इससे हमें हमारा पूरा ब्लॉक दिखाई देगा। ठीक है? यह ऐसा एक-एक लाइन का इसलिए दिख रहा है बिकॉज़ यह हॉरिजॉन्टली स्क्रबल है। ठीक है? दिस इज़ अ प्रॉपर ब्लॉक जो जनरेट हुआ है। ठीक है? तो आई होप आपको समझ में आ गया कि कैसे आप प्र्ट चेनिंग कर सकते हो यूजिंग लैंग ग्राफ। ऑलरेडी आपको एक बेनिफिट दिख रहा होगा। अगर यह सेम काम आप लैंग चेन की हेल्प से करते चेन बना करके। तो वहां पर जब आप फाइनल स्टेट में आउटपुट निकालते तो आपको सिर्

In [ ]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [ ]:
final_prompt

StringPromptValue(text="\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      रेट माय ब्लॉग। ठीक है? और जनरेट अ स्कोर इंटीजर स्कोर। ठीक है? तो आपको स्टेट में चेंजज़ करने पड़ेंगे और आपको वर्क फ्लो में भी थोड़े चेंजज़ करने पड़ेंगे। ये करके देखना आप। अह बहुत खास कुछ इंप्रूवमेंट नहीं है। बट सिंस आप लोग पहली बार कर रहे हो तो ये छोटी-छोटी चीजें भी अगर आप खुद से कर पा रहे हो तो इट इज़ सिग्निफिकेंट प्रोग्रेस। ठीक है? तो अह विथ दैट आई विल कंक्लूड दिस वीडियो। अगर आपको वीडियो पसंद आया तो प्लीज लाइक करना। अगर आपने चैनल को सब्सक्राइब नहीं किया है, प्लीज डू सब्सक्राइब। मिलते हैं नेक्स्ट वीडियो में। बाय।\n\nफ्यूचर आउटलुक, कंक्लूजन। अब हम लोग लिखते हैं फाइनल स्टेट कंटेंट प्रिंट। इससे हमें हमारा पूरा ब्लॉक दिखाई देगा। ठीक है? यह ऐसा एक-एक लाइन का इसलिए दिख रहा है बिकॉज़ यह हॉरिजॉन्टली स्क्रबल है। ठीक है? दिस इज़ अ प्रॉपर ब्लॉक जो जनरेट हुआ है। ठीक है? तो आई होप आपको समझ में आ गया कि कैसे आप प्र्ट चे

## Step 4 - Generation

In [ ]:
answer = llm.invoke(final_prompt)
print(answer.content)

No, the topic of new birth policy in china is not discussed in the provided transcript context.


## Building a Chain

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [ ]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [ ]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [ ]:
# parallel_chain.invoke('who is Demis')

In [ ]:
parser = StrOutputParser()

In [ ]:
main_chain = parallel_chain | prompt | llm | parser

In [ ]:
main_chain.invoke('Can you summarize the video')

'वीडियो में नितेश ने अपने एजेंटिक एआई यूजिंग लंग ग्राफ प्लेलिस्ट को जारी रखा है। उन्होंने अपने पिछले वीडियो में कवर किए गए विषयों की रिवाइज की और बताया कि उन्होंने जनरेटिव एआई और एजेंटिक एआई के बीच मुख्य अंतरों पर चर्चा की, एजेंटिक एआई के बारे में विस्तार से जानकारी दी, और लैंग ग्राफ की जरूरत को समझाया। उन्होंने एक यूज केस के माध्यम से दोनों लाइब्रेरीज के बीच के अंतरों को समझाया।\n\nअब वे अपने वर्कफ्लो को बनाने के लिए आगे बढ़ रहे हैं। उन्होंने अपने वर्कफ्लो को एक आउटलाइन में व्यवस्थित किया है और इसमें टाइटल, इंट्रोडक्शन, हिस्टोरिकल कॉन्टेक्स्ट, करंट स्टेट, चैलेंजेस और अपोर्चुनिटीज, फ्यूचर आउटलुक, और कंक्लूजन शामिल हैं।\n\nउन्होंने अपने वर्कफ्लो को एक ब्लॉग पोस्ट के रूप में लिखने की योजना बनाई है और इसमें उन्होंने अपने वर्कफ्लो को एक स्टेट ग्राफ के रूप में प्रस्तुत किया है। उन्होंने अपने वर्कफ्लो को एक क्लास में डिफाइन करने की योजना बनाई है जो टाइप्ड डिक्शनरी का एक्सटेंशन होगा।\n\nउन्होंने अपने वर्कफ्लो को एक एलएलएम के साथ एक्सीक्यूट करने की योजना बनाई है और इसमें उन्होंने अपने प्र्प को